# Fast-ion orbits

The local slowing down (`slow_down`) keeps every fast ion on its birth surface. `follow_orbits` follows the births instead as guiding centres, with Monte Carlo Coulomb collisions that use the same drags, Coulomb logarithms and end energy, until they thermalise or cross the wall: confinement, losses and wall loads. The orbits are pushed in physical space by RK4 on the cubic-spline field of a `GriddedEquilibrium`, in a compiled kernel; the drifts follow from the exact derivatives of the interpolated field.

This page checks the pusher in a tokamak (energy and canonical momentum, the width of a passing orbit), compares the orbit-following moments with the local slowing down where the orbits are narrow, and then follows the births of the stand-in W7-X injectors (see [W7-X deposition](/freeofcharge/examples/w7x-deposition/)) in DESC's W7-X equilibrium to a W7-X-like wall. **The wall is a uniform 10 cm offset of the plasma boundary, not the W7-X wall, and the injector geometry is a stand-in**: the losses and wall loads show what the code computes, not W7-X's.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import plasma_plots as pp
import xarray as xr
from plasmacoll.constants import ATOMIC_MASS, ELEMENTARY_CHARGE

import freeofcharge as foc
from freeofcharge import orbits

pp.set_backend("plotly")
pp.set_math("latex")
warnings.filterwarnings("ignore", category=UserWarning, module="desc")

## Orbits in a tokamak

A Solov'ev tokamak ($R_0 = 3$ m, $a = 0.6$ m, elongation 1.5, 3 T) sampled on a grid that covers its wall (`orbit_field`), and five 55 keV protons started on the outboard midplane with pitches from $-0.9$ to $0.95$. `guiding_centres` turns a particle (position and velocity) into a guiding centre: the position one Larmor radius off, the parallel velocity and the magnetic moment. `trace_orbits` pushes them without collisions and returns the orbits in plasma-plots' layout, $(t, \mathrm{marker})$ variables.

In [ ]:
tokamak = foc.Equilibrium.solovev(3.0, 0.6, 1.5, 3.0, 1.2, wall_psi_n=1.3)
field = orbits.orbit_field(tokamak, (128, 8, 128), box=((2.0, 3.9), (-1.25, 1.25)))
pitch = np.array([-0.9, -0.3, 0.1, 0.5, 0.95])
start = np.tile([3.3, 0.0, 0.0], (5, 1))
b = np.asarray(field.field_at(start))
unit = b / np.linalg.norm(b, axis=1)[:, None]
across = np.cross(unit, [0.0, 0.0, 1.0])
across /= np.linalg.norm(across, axis=1)[:, None]
speed = np.sqrt(2 * 55e3 * ELEMENTARY_CHARGE / ATOMIC_MASS)
velocity = speed * (pitch[:, None] * unit + np.sqrt(1 - pitch**2)[:, None] * across)
markers = orbits.guiding_centres(start, velocity, np.ones(5), field, ATOMIC_MASS)
traces = orbits.trace_orbits(markers, field, 2e-3, samples=2001, step=0.1)
traces

The poloidal projection: the deeply trapped ions (pitch 0.1 and $-0.3$) bounce on bananas, the passing ones circle the axis, shifted off their surface by the drift.

In [ ]:
with pp.figure(figsize=(5, 6)) as fig:
    ax = fig[0]
    boundary = tokamak.boundary.vertices
    ax.plot(boundary[:, 0], boundary[:, 1], color="black", linewidth=1)
    ax.plot(
        tokamak.wall.vertices[:, 0],
        tokamak.wall.vertices[:, 1],
        color="gray",
        linewidth=1,
    )
    for k, p in enumerate(pitch):
        ax.plot(
            traces.R.values[:, k],
            traces.z.values[:, k],
            linewidth=1,
            label=f"pitch {p:+.2f}",
        )
    ax.set_aspect("equal")
    ax.set_xlabel("$R$ [m]")
    ax.set_ylabel("$Z$ [m]")
    ax.set_title("Guiding-centre orbits, 2 ms")
    ax.legend()

Two invariants test the integration over hundreds of bounce and transit periods: the kinetic energy $m(\tfrac12 v_\parallel^2 + \mu B)$ and, in an axisymmetric field, the canonical toroidal momentum $p_\varphi = m R v_\parallel b_\varphi + e\psi$. The energy error falls as the fifth power of the step length (RK4); $p_\varphi$ is kept to the difference between the interpolated field and the flux $\psi$ of the equilibrium. The magnetic moment is a constant of the guiding-centre equations, so it needs no check.

In [ ]:
psi_span = ELEMENTARY_CHARGE * (tokamak.psi_boundary - tokamak.psi_axis)
for step in (0.2, 0.1, 0.05):
    run = orbits.trace_orbits(markers, field, 2e-3, samples=201, step=step)
    energy = run.energy.values
    r, z = run.R.values, run.z.values
    b = np.asarray(field.field_at(np.stack([run.x.values, run.y.values, z], -1)))
    b_phi = (-run.y.values * b[..., 0] + run.x.values * b[..., 1]) / r
    p_phi = ATOMIC_MASS * r * run.v_parallel.values * b_phi / run.B.values
    p_phi += ELEMENTARY_CHARGE * tokamak.psi(r, z)
    print(
        f"step {step * 100:4.0f} cm: largest energy error {np.max(np.abs(energy / energy[0] - 1)):.1e}, "
        f"p_phi {np.max(np.abs(p_phi - p_phi[0])) / psi_span:.1e} of e (psi_b - psi_0)"
    )

## The width of a passing orbit

On a large-aspect-ratio tokamak ($R_0 = 10$ m, $a = 1$ m, 2 T, $q_0 = 1.5$), a passing orbit crosses the midplane outboard and inboard on different flux surfaces. The difference is fixed exactly by the conservation of $p_\varphi$, and estimated by $2 q \rho_\parallel$ with $\rho_\parallel = m v_\parallel / (eB)$, which neglects the variation of $v_\parallel$ and $B$ around the orbit (terms of the order of the inverse aspect ratio and $(1 - \xi^2)/\xi^2$). The label of a surface on the midplane is $u = (R^2 - R_0^2) / (2 R_0)$, the same inboard and outboard for the Solov'ev flux.

In [ ]:
r0, b0, q0 = 10.0, 2.0, 1.5
large = foc.Equilibrium.solovev(r0, 1.0, 1.0, b0, q0, wall_psi_n=None, points=257)
large_field = orbits.orbit_field(large, (256, 4, 256))
shells = large.flux_surfaces(np.linspace(0, 1, 41))
rows = []
for xi in (0.95, 0.9, 0.6, -0.9):
    p0 = np.array([[r0 + 0.5, 0.0, 0.0]])
    b = np.asarray(large_field.field_at(p0))[0]
    u_b = b / np.linalg.norm(b)
    e1 = np.cross(u_b, [0, 0, 1.0])
    e1 /= np.linalg.norm(e1)
    v = speed * (xi * u_b + np.sqrt(1 - xi**2) * e1)
    gc = orbits.guiding_centres(p0, v[None], np.ones(1), large_field, ATOMIC_MASS)
    run = orbits.trace_orbits(gc, large_field, 3e-4, samples=30001, step=0.02)
    r, z, vp = (run[k].values[:, 0] for k in ("R", "z", "v_parallel"))
    k = np.nonzero(np.sign(z[1:]) != np.sign(z[:-1]))[0]
    t = z[k] / (z[k] - z[k + 1])
    rc, vc = r[k] + t * (r[k + 1] - r[k]), vp[k] + t * (vp[k + 1] - vp[k])
    uc = (rc**2 - r0**2) / (2 * r0)
    bc = np.linalg.norm(
        np.asarray(large_field.field_at(np.stack([rc, 0 * rc, 0 * rc], -1))), axis=-1
    )
    out, inn = uc > 0, uc < 0
    width = uc[out].mean() + uc[inn].mean()
    r_mid = 0.5 * (uc[out].mean() - uc[inn].mean())
    exact = (
        (ATOMIC_MASS * b0 * r0 / ELEMENTARY_CHARGE)
        * (vc[inn].mean() / bc[inn].mean() - vc[out].mean() / bc[out].mean())
        / (2 * b0 / (2 * r0**2 * q0) * r0**2 * r_mid)
    )
    q = np.interp(float(large.rho(r0 + r_mid, 0.0)), shells.rho, shells.safety_factor)
    estimate = 2 * q * ATOMIC_MASS * np.abs(vc).mean() / (ELEMENTARY_CHARGE * b0)
    rows.append((xi, 100 * width, 100 * exact, 100 * estimate, abs(width) / estimate))
print("pitch   width [cm]   from p_phi [cm]   2 q rho_par [cm]   width / estimate")
for xi, width, exact, estimate, ratio in rows:
    print(
        f"{xi:+5.2f}   {width:+9.3f}   {exact:+14.3f}   {estimate:16.3f}   {ratio:12.3f}"
    )

The measured width equals the $p_\varphi$ prediction to $10^{-4}$; the estimate $2q\rho_\parallel$ holds to about 10% for the well-passing orbits and fails, as it should, towards the trapped-passing boundary.

## Collisions against the local slowing down

`collide` acts on each marker between pushes: the drag $dv/dt = -a_e v - a_i / v^2$ of `slow_down` exactly over the step, energy diffusion that keeps a Maxwellian stationary, and Lorentz pitch-angle scattering with `slow_down`'s deflection rate. Where the orbits are narrow, the time integrals along the orbits must give `slow_down`'s heating, fast-ion number and energy: a 10 T Solov'ev tokamak (orbit widths below a centimetre), $10^{20}\,\mathrm{m^{-3}}$ and 1.5 keV, 55 keV hydrogen, the same end energy ($2 T_i$), without energy diffusion (which `slow_down` leaves out).

In [ ]:
high = foc.Equilibrium.solovev(3.0, 0.6, 1.5, 10.0, 1.2, wall_psi_n=None)
rho_edges = np.linspace(0, 1, 11)
high_shells = high.flux_surfaces(rho_edges)
high_field = orbits.orbit_field(high, (128, 4, 128))
origin, direction = foc.tangential_injection(2.9, 6.0)
beam = foc.NeutralBeam.gaussian(
    origin,
    direction,
    55e3,
    1e6,
    mass_number=1,
    width=0.1,
    divergence=0.005,
    markers=400,
    seed=1,
)
plasma = foc.Plasma.parabolic(1e20, 1.5e3, ions={"H": (1, 1, 1.0)})
births = foc.deposit(
    beam, high, plasma, step=0.01, births_per_marker=1, rho_edges=rho_edges, seed=2
)
local = foc.slow_down(
    births, plasma, high_shells, orbit_average=False, thermal_factor=2.0
)
start = time.perf_counter()
followed = orbits.follow_orbits(
    births,
    high_field,
    plasma,
    surfaces=high_shells,
    step=0.2,
    thermal_factor=2.0,
    energy_diffusion=False,
    seed=3,
)
print(
    f"{len(followed.markers.marker)} markers in {time.perf_counter() - start:.1f} s; power balance {followed.power_balance:+.1e}"
)
p = local.birth_power
print(
    f"electrons: orbits {followed.electron_heating.sum() / p:.4f}, local {local.electron_heating.sum() / p:.4f} of the birth power"
)
print(
    f"ions:      orbits {followed.ion_heating.sum() / p:.4f}, local {local.ion_heating.sum() / p:.4f}"
)
print(
    f"thermalised: orbits {followed.thermalised_power / p:.4f}, local {local.thermalised_power / p:.4f}"
)
print(
    f"fast ions: orbits / local {followed.particles.sum() / np.sum(local.fast_ion_density * high_shells.volume):.4f}"
)
print(
    f"fast-ion energy: orbits / local {followed.stored_energy / local.stored_energy:.4f}"
)

In [ ]:
both = xr.concat(
    [
        followed.to_dataset().fast_ion_density,
        local.to_dataset(high_shells).fast_ion_density,
    ],
    dim=xr.DataArray(["orbits", "local"], dims="model", name="model"),
)
both.attrs.update(long_name="fast-ion density", units="1/m^3")
both.plasma.plot.profiles(x="rho", over="model")

Shell by shell the two differ by the Monte Carlo noise of 400 markers (the pitch scattering moves ions between trapped and passing orbits, so their surfaces wander a little); the totals agree to about 1%. With energy diffusion the electron drag becomes $-2a_e(E - \tfrac32 T_e)$ (the standard result), so the ions slow down more slowly near the end: at these temperatures the fast-ion energy is about 5% higher, a physical difference `slow_down` does not have.

## W7-X: the births

DESC's W7-X on the cached grid of [W7-X deposition](/freeofcharge/examples/w7x-deposition/) ($128 \times 64 \times 128$ nodes over a field period), the two OP1.2b sources at $n_{e0} = 4\times10^{19}\,\mathrm{m^{-3}}$, $T_0 = 2$ keV, and simgeometry's stand-in wall, the boundary offset by 10 cm (`w7x_like_wall`). The grid's field outside $\rho = 1$ is DESC's continuation (the field of the nearest boundary point), not a vacuum field: with coils, `CoilField` gives the coils' field there and `orbit_field` samples it.

In [ ]:
from desc.grid import LinearGrid  # noqa: F401  (DESC only for the cached grid)
from simgeometry.data import w7x_like_wall

from freeofcharge.desc_equilibrium import DESCEquilibrium
from freeofcharge.injectors import w7x_injectors

cache = Path(
    os.environ.get("FREEOFCHARGE_CACHE", Path.home() / ".cache" / "freeofcharge")
)
w7x = foc.GriddedEquilibrium.cached(
    cache / "w7x_desc_128_64_128.nc",
    DESCEquilibrium.from_example("W7-X"),
    (128, 64, 128),
)
wall = w7x_like_wall(0.10)
plasma = foc.Plasma.parabolic(
    4e19, 2e3, ions={"H": (1, 1, 1.0)}, radial_coordinate="toroidal"
)
op12b = w7x_injectors(sources={"NI21": [1, 2]})
rho_edges = np.linspace(0, 1, 11)
deposition = foc.deposit_sources(
    op12b, w7x, plasma, markers=170, seed=1, step=0.01, rho_edges=rho_edges
).total()
print(
    f"{deposition.births.weights.shape[0]} births, {deposition.absorbed_power / 1e6:.2f} MW absorbed"
)
print(
    f"birth pitch from {np.min(deposition.birth_pitch):.2f} to {np.max(deposition.birth_pitch):.2f}"
)

## Prompt losses

Without collisions for 1 ms (tens of toroidal transits), the births pass the field in the stand-in geometry (pitch about $-0.6$ to $-1$, against the field): none of them reaches the wall. Prompt losses here would need more perpendicular injection or a wall closer to the plasma.

The energy is kept to about $10^{-5}$ for most markers; the largest errors, about 1%, belong to orbits that graze $\rho = 1$, where the grid's field has a kink (DESC's continuation outside); shorter steps (5 cm) reduce them by more than an order of magnitude.

In [ ]:
start = time.perf_counter()
prompt = orbits.follow_orbits(
    deposition, w7x, None, wall=wall, step=0.1, max_time=1e-3, interval=5e-5
)
print(
    f"collisionless 1 ms: {time.perf_counter() - start:.0f} s; fates {prompt.fate_fractions()}"
)
error = np.abs(prompt.markers.end_energy / prompt.markers.birth_energy - 1)
print(
    f"energy error after 1 ms: median {np.median(error):.1e}, largest {error.max():.1e}"
)

## Slowing down with orbits: losses and wall loads

With collisions, pitch-angle scattering moves ions onto trapped orbits, whose drifts are not confined in this configuration at this energy: some leave the plasma and hit the wall. All ~1000 markers are followed until they thermalise (below $2 T_i$) or are lost, a few slowing-down times (up to about 40 ms). The kernel takes about 0.35 µs per step on four cores, with steps of 10 cm.

In [ ]:
start = time.perf_counter()
slowing = orbits.follow_orbits(
    deposition,
    w7x,
    plasma,
    surfaces=w7x.flux_surfaces(rho_edges),
    wall=wall,
    step=0.1,
    seed=1,
)
elapsed = time.perf_counter() - start
print(
    f"{len(slowing.markers.marker)} markers in {elapsed:.0f} s, up to {slowing.markers.time.max().item() * 1e3:.0f} ms"
)
print(
    "fates (share of the births):",
    {k: round(v, 4) for k, v in slowing.fate_fractions().items() if v},
)
print(
    "power (share of the birth power):",
    {k: round(v, 4) for k, v in slowing.power_fractions().items()},
)
print(f"power balance {slowing.power_balance:+.1e}")

The heating and the fast-ion density against the local slowing down of the same births on the same 3D shells (`slow_down` with the local pitch): the orbits move fast ions outward and lose a few percent of the power.

In [ ]:
local_w7x = foc.slow_down(
    deposition,
    plasma,
    w7x.flux_surfaces(rho_edges),
    orbit_average=False,
    thermal_factor=2.0,
    trapped_electron_correction=lambda surfaces, plasma: np.zeros(len(rho_edges) - 1),
)
compare = xr.concat(
    [
        slowing.to_dataset().heating_density.sum("species"),
        local_w7x.to_dataset(w7x.flux_surfaces(rho_edges)).heating_density.sum(
            "species"
        ),
    ],
    dim=xr.DataArray(["orbits", "local"], dims="model", name="model"),
)
compare.attrs.update(long_name="heating", units="W/m^3")
compare.plasma.plot.profiles(x="rho", over="model").show()
density = xr.concat(
    [slowing.to_dataset().fast_ion_density, local_w7x.to_dataset().fast_ion_density],
    dim=xr.DataArray(["orbits", "local"], dims="model", name="model"),
)
density.attrs.update(long_name="fast-ion density", units="1/m^3")
density.plasma.plot.profiles(x="rho", over="model")

Where the lost ions hit the wall: `wall_loads` sums the power of each loss on the face it crosses and divides by the face area (W/m², a steady state since the weights are ions per second); on a `FourierSurface` it also bins the load on the wall's angles $(\theta, \varphi)$.

In [ ]:
loads = slowing.wall_loads(theta_bins=48, phi_bins=180)
print(
    f"lost {loads.attrs['lost_power'] / 1e3:.1f} kW, peak load {loads.attrs['peak_wall_load'] / 1e3:.1f} kW/m^2 on one face"
)
lost = slowing.losses()
print(
    f"lost markers: {lost.sizes['marker']}, mean energy at the wall {lost.end_energy.mean().item() / 1e3:.1f} keV, median time {np.median(lost.time) * 1e3:.2f} ms"
)
heat = loads.wall_load_map.fillna(0.0) / 1e3
heat.attrs.update(long_name="wall load", units="kW/m^2")
heat.plasma.plot.view(x="phi", y="theta")

## What ASCOT5 should check

ASCOT5 is not available here. The comparisons that remain, on the same inputs (`freeofcharge.ascot5` writes the tokamak case; for W7-X, DESC's equilibrium as `B_3DS` and the wall as `wall_3D`):

- the tokamak benchmark of [ASCOT5 benchmark](/freeofcharge/benchmarks/ascot5/), now with orbits in freeofcharge too: the fast-ion current, the heating split and the thermalised power against ASCOT5's run with orbits;
- W7-X wall loads and fast-ion confinement against Äkäslompolo et al., Nucl. Fusion 58, 082010 (2018) (NBI wall loads in W7-X with ASCOT) and the 2019 W7-X fast-ion work, once the real wall (IPP's CAD) and the real injector geometry replace the stand-ins;
- the orbit benchmark of PolarFEEC (logical against physical coordinates on the same W7-X equilibrium).